In [2]:
!find /kaggle/input -maxdepth 3 -name "test.jsonl"
!ls /kaggle/input

/kaggle/input/competitions/otto-recommender-system/test.jsonl
competitions


In [6]:
import json, collections

TEST = "/kaggle/input/competitions/otto-recommender-system/test.jsonl"

cnt = collections.Counter()
n_sessions = 0
with open(TEST) as f:
    for line in f:
        s = json.loads(line)
        n_sessions += 1
        for e in s["events"]:
            cnt[e["aid"]] += 1

popular = [aid for aid, _ in cnt.most_common(20)]
print(n_sessions, "sessions")
print(popular)

1671803 sessions
[485256, 1460571, 108125, 986164, 554660, 1551213, 754412, 832192, 33343, 579690, 1006198, 688602, 329725, 29735, 184976, 1019736, 1022566, 1043508, 496180, 944778]


In [8]:
def uniq_recent(aids):
    seen, out = set(), []
    for a in reversed(aids):
        if a not in seen:
            seen.add(a); out.append(a)
    return out

def top20(aids):
    out = uniq_recent(aids)[:20]
    if len(out) < 20:
        s = set(out)
        for p in popular:
            if p not in s:
                out.append(p); s.add(p)
                if len(out) == 20: break
    return out

with open(TEST) as f, open("submission.csv", "w") as w:
    w.write("session_type,labels\n")
    for line in f:
        s = json.loads(line)
        ev = s["events"]
        sid = s["session"]
        all_aids = [e["aid"] for e in ev]
        buy_aids = [e["aid"] for e in ev if e["type"] in ("carts", "orders")]

        clicks = top20(all_aids)

        buys_first = uniq_recent(buy_aids)
        buys_set = set(buys_first)
        rest = [a for a in uniq_recent(all_aids) if a not in buys_set]
        co = top20(list(reversed(buys_first + rest)))

        w.write(f"{sid}_clicks,{' '.join(map(str, clicks))}\n")
        w.write(f"{sid}_carts,{' '.join(map(str, co))}\n")
        w.write(f"{sid}_orders,{' '.join(map(str, co))}\n")

In [9]:
!head -4 submission.csv
!wc -l submission.csv
!ls -lh submission.csv

session_type,labels
12899779_clicks,59625 485256 1460571 108125 986164 554660 1551213 754412 832192 33343 579690 1006198 688602 329725 29735 184976 1019736 1022566 1043508 496180
12899779_carts,59625 485256 1460571 108125 986164 554660 1551213 754412 832192 33343 579690 1006198 688602 329725 29735 184976 1019736 1022566 1043508 496180
12899779_orders,59625 485256 1460571 108125 986164 554660 1551213 754412 832192 33343 579690 1006198 688602 329725 29735 184976 1019736 1022566 1043508 496180
5015410 submission.csv
-rw-r--r-- 1 root root 767M Nov 25 08:06 submission.csv
